Recreation of old article
Python 3.11
    numpy
    pandas
    matplotlib
    collections
    typing

In [26]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Set
from collections import Counter
import os
import sys
import re
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [27]:
#Load the dataset
training_test = pd.read_excel('dataset.xlsx', sheet_name=0)
validation = pd.read_excel('dataset.xlsx', sheet_name=1)

print(f"Loaded dataset: {training_test.shape[0]} molecules in training_test and {validation.shape[0]} molecules in validation")
#training_test.head()

train_df = training_test[training_test['Status'] == 'Train']
test_df = training_test[training_test['Status'] == 'Test']
print(f"Training set: {train_df.shape[0]} molecules, Test set: {test_df.shape[0]} molecules")

y_test = test_df['Class']
X_test = test_df.drop(columns=['Status', 'Class', 'Smiles', 'CAS-RN'])

y_train = train_df['Class']
X_train = train_df.drop(columns=['Status', 'Class', 'Smiles', 'CAS-RN'])


y_validation = validation['class']
X_validation = validation.drop(columns=['class', 'Smiles', 'CAS-RN'])

column_order = X_train.columns.tolist()
X_validation = X_validation.reindex(columns=column_order)

print(column_order)
print(X_validation.columns.tolist())

Loaded dataset: 1055 molecules in training_test and 670 molecules in validation
Training set: 837 molecules, Test set: 218 molecules
['SpMax_L', 'J_Dz(e)', 'nHM', 'F01[N-N]', 'F04[C-N]', 'NssssC', 'nCb-', 'C%', 'nCp', 'nO', 'F03[C-N]', 'SdssC', 'HyWi_B(m)', 'LOC', 'SM6_L', 'F03[C-O]', 'Me', 'Mi', 'nN-N', 'nArNO2', 'nCRX3', 'SpPosA_B(p)', 'nCIR', 'B01[C-Br]', 'B03[C-Cl]', 'N-073', 'SpMax_A', 'Psi_i_1d', 'B04[C-Br]', 'SdO', 'TI2_L', 'nCrt', 'C-026', 'F02[C-N]', 'nHDon', 'SpMax_B(m)', 'Psi_i_A', 'nN', 'SM6_B(m)', 'nArCOOR', 'nX']
['SpMax_L', 'J_Dz(e)', 'nHM', 'F01[N-N]', 'F04[C-N]', 'NssssC', 'nCb-', 'C%', 'nCp', 'nO', 'F03[C-N]', 'SdssC', 'HyWi_B(m)', 'LOC', 'SM6_L', 'F03[C-O]', 'Me', 'Mi', 'nN-N', 'nArNO2', 'nCRX3', 'SpPosA_B(p)', 'nCIR', 'B01[C-Br]', 'B03[C-Cl]', 'N-073', 'SpMax_A', 'Psi_i_1d', 'B04[C-Br]', 'SdO', 'TI2_L', 'nCrt', 'C-026', 'F02[C-N]', 'nHDon', 'SpMax_B(m)', 'Psi_i_A', 'nN', 'SM6_B(m)', 'nArCOOR', 'nX']


In [28]:
def measurements(pred, truth, positive="RB", negative="NRB"):
    pred = np.array(pred)
    truth = np.array(truth)

    # Confusion matrix components
    TP = np.sum((pred == positive) & (truth == positive))
    TN = np.sum((pred == negative) & (truth == negative))
    FP = np.sum((pred == positive) & (truth == negative))
    FN = np.sum((pred == negative) & (truth == positive))

    # Avoid division by zero
    Sn = TP / (TP + FN) if (TP + FN) > 0 else 0
    Sp = TN / (TN + FP) if (TN + FP) > 0 else 0

    # Non-error rate
    NER = (Sn + Sp) / 2

    # Error rate
    ER = 1 - NER

    return {
        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "Sensitivity (Sn)": Sn,
        "Specificity (Sp)": Sp,
        "NER": NER,
        "ER": ER
    }

def summarize_metrics(metrics_knn, metrics_svm, metrics_pls, results_c1, results_c2, coverage):
    summary = pd.DataFrame({
    "kNN": {
        "ER": metrics_knn["ER"],
        "Sn": metrics_knn["Sensitivity (Sn)"],
        "Sp": metrics_knn["Specificity (Sp)"]
        
    },
    "SVM": {
        "ER": metrics_svm["ER"],
        "Sn": metrics_svm["Sensitivity (Sn)"],
        "Sp": metrics_svm["Specificity (Sp)"]
    },
    "PLSDA": {
        "ER": metrics_pls["ER"], 
        "Sn": metrics_pls["Sensitivity (Sn)"],
        "Sp": metrics_pls["Specificity (Sp)"]
    },
    "Consensus 1": {
        "ER": results_c1["ER"],
        "Sn": results_c1["Sensitivity (Sn)"],
        "Sp": results_c1["Specificity (Sp)"]
    },
    "Consensus 2": {
        "ER": results_c2["ER"],
        "Sn": results_c2["Sensitivity (Sn)"],
        "Sp": results_c2["Specificity (Sp)"]
    }
})
    summary = summary.T.astype(float).round(2)  # models as rows
    summary["% Not Classified"] = None
    summary.loc["Consensus 2", "% Not Classified"] = round(coverage, 2)
    return summary

In [29]:
def knn_data_filtering(train, test):
    knn_descriptors = [
    "C%",
    "F01[N-N]",
    "F03[C-N]",
    "F04[C-N]",
    "J_Dz(e)",
    "nCb-",
    "nCp",
    "nHM",
    "nO",
    "NssssC",
    "SdssC",
    "SpMax_L"
]
    train_filtered = train[knn_descriptors]
    test_filtered = test[knn_descriptors]
    return train_filtered, test_filtered

def svm_data_filtering(train, test):
    svm_descriptors = [
    "C-026",
    "F02[C-N]",
    "nArCOOR",
    "nCb-",
    "nCrt",
    "nHDon",
    "nN",
    "nN-N",
    "nX",
    "Psi_i_A",
    "SM6_B(m)",
    "SpMax_B(m)",
    "SpMax_L"
]
    train_filtered = train[svm_descriptors]
    test_filtered = test[svm_descriptors]
    return train_filtered, test_filtered

def plsda_data_filtering(train, test):
    plsda_descriptors = [
    "B01[C-Br]",
    "B03[C-Cl]",
    "B04[C-Br]",
    "C%",
    "F03[C-O]",
    "F04[C-N]",
    "HyWi_B(m)",
    "LOC",
    "Me",
    "Mi",
    "N-073",
    "nArNO2",
    "nCIR",
    "nCRX3",
    "nN-N",
    "nO",
    "Psi_i_1d",
    "SdO",
    "SM6_L",
    "SpMax_A",
    "SpMax_L",
    "SpPosA_B(p)",
    "TI2_L"
]
    train_filtered = train[plsda_descriptors]
    test_filtered = test[plsda_descriptors]
    return train_filtered, test_filtered


In [30]:
##Knn

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, recall_score

# Create model
knn = KNeighborsClassifier(
    n_neighbors=6,
    metric='euclidean',     # default
    weights='uniform'       # or 'distance'
)
#Data
X_train_knn, X_test_knn = knn_data_filtering(X_train, X_test)
print("num featuers used for KNN:", X_train_knn.columns.tolist().__len__())

# Train
knn.fit(X_train_knn, y_train)

# Predict
pred_knn = knn.predict(X_test_knn)

# Evaluate
metrics_knn = measurements(pred_knn, y_test)

print("KNN Metrics:")
for key, value in metrics_knn.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

num featuers used for KNN: 12
KNN Metrics:
TP: 53
TN: 128
FP: 18
FN: 19
Sensitivity (Sn): 0.7361
Specificity (Sp): 0.8767
NER: 0.8064
ER: 0.1936


In [31]:
#SVM

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

SVM_model = Pipeline([
    ('scaler', StandardScaler()),  # mandatory for SVM
    ('svm', SVC(kernel='rbf', C=5.0, gamma='scale'))
])

#Data
X_train_svm, X_test_svm = svm_data_filtering(X_train, X_test)
print("num featuers used for SVM:", X_train_svm.columns.tolist().__len__())


SVM_model.fit(X_train_svm, y_train)

pred_svm = SVM_model.predict(X_test_svm)
metrics_svm = measurements(pred_svm, y_test)
print("SVM Metrics:")
for key, value in metrics_svm.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

num featuers used for SVM: 13
SVM Metrics:
TP: 52
TN: 132
FP: 14
FN: 20
Sensitivity (Sn): 0.7222
Specificity (Sp): 0.9041
NER: 0.8132
ER: 0.1868


In [32]:
#PLSDA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

PLSDA_model = Pipeline([
    ('scaler', StandardScaler()),  # mandatory for SVM
    ('svm', SVC(kernel='rbf', C=5.0, gamma='scale'))
])

#Data
X_train_plsda, X_test_plsda = plsda_data_filtering(X_train, X_test)
print("num featuers used for PLSDA:", X_train_plsda.columns.tolist().__len__())

PLSDA_model.fit(X_train_plsda, y_train)

pred_pls = PLSDA_model.predict(X_test_plsda)
metrics_pls = measurements(pred_pls, y_test)
print("PLS-DA Metrics:")
for key, value in metrics_pls.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

num featuers used for PLSDA: 23
PLS-DA Metrics:
TP: 53
TN: 137
FP: 9
FN: 19
Sensitivity (Sn): 0.7361
Specificity (Sp): 0.9384
NER: 0.8372
ER: 0.1628


In [33]:
# predictions from three models
df_preds = pd.DataFrame({
    'KNN': pred_knn,
    'SVM': pred_svm,
    'PLS': pred_pls
})
def consensus1_prediction(preds_df):
    return preds_df.mode(axis=1)[0]
combined_pred = consensus1_prediction(df_preds)
results_c1 = measurements(combined_pred, y_test)
print("Combined Metrics (Consensus 1):")
for key, value in results_c1.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

Combined Metrics (Consensus 1):
TP: 54
TN: 136
FP: 10
FN: 18
Sensitivity (Sn): 0.7500
Specificity (Sp): 0.9315
NER: 0.8408
ER: 0.1592


In [34]:
def consensus2_prediction(preds_df):
    return preds_df.apply(
        lambda row: row.iloc[0] if row.nunique() == 1 else np.nan,
        axis=1
    )
combined_pred = consensus2_prediction(df_preds)
y_test = np.array(y_test)

mask = combined_pred.notna()
y_test_filtered = y_test[mask]
combined_filtered = combined_pred[mask]

coverage = 1-mask.mean()
print(f"not assigned: {coverage:.2%}")

results_c2 = measurements(combined_filtered, y_test_filtered)
print("Combined Metrics (Consensus 2):")
for key, value in results_c2.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

not assigned: 17.89%
Combined Metrics (Consensus 2):
TP: 42
TN: 121
FP: 6
FN: 10
Sensitivity (Sn): 0.8077
Specificity (Sp): 0.9528
NER: 0.8802
ER: 0.1198


# Test set

In [35]:
summary = summarize_metrics(metrics_knn, metrics_svm, metrics_pls, results_c1, results_c2, coverage)
print("\nSummary of Metrics for test data:")
print(summary)


Summary of Metrics for test data:
               ER    Sn    Sp % Not Classified
kNN          0.19  0.74  0.88             None
SVM          0.19  0.72  0.90             None
PLSDA        0.16  0.74  0.94             None
Consensus 1  0.16  0.75  0.93             None
Consensus 2  0.12  0.81  0.95             0.18


# Validation set

In [36]:
_, knn_data_filtered= knn_data_filtering(X_train, X_validation)
val_pred_knn = knn.predict(knn_data_filtered)
knn_metrics_val = measurements(val_pred_knn, y_validation)

_, svm_data_filtered = svm_data_filtering(X_train, X_validation)
val_pred_svm = SVM_model.predict(svm_data_filtered)
svm_metrics_val = measurements(val_pred_svm, y_validation) 

_, plsda_data_filtered = plsda_data_filtering(X_train, X_validation)
val_pred_pls = PLSDA_model.predict(plsda_data_filtered)
pls_metrics_val = measurements(val_pred_pls, y_validation)

consensus1_val_pred = consensus1_prediction(pd.DataFrame({
    'KNN': val_pred_knn,
    'SVM': val_pred_svm,
    'PLS': val_pred_pls
}))
consensus1_metrics_val = measurements(consensus1_val_pred, y_validation)

consensus2_val_pred = consensus2_prediction(pd.DataFrame({
    'KNN': val_pred_knn,
    'SVM': val_pred_svm,
    'PLS': val_pred_pls
}))

y_validation = np.array(y_validation)

mask = consensus2_val_pred.notna()
y_validation_filtered = y_validation[mask]
consensus2_val_filtered = consensus2_val_pred[mask]

coverage = 1-mask.mean()

consensus2_metrics_val = measurements(consensus2_val_filtered, y_validation_filtered)
summary_val = summarize_metrics(knn_metrics_val, svm_metrics_val, pls_metrics_val, consensus1_metrics_val, consensus2_metrics_val, coverage)
print("\nSummary of Metrics for validation data:")
print(summary_val)


Summary of Metrics for validation data:
               ER    Sn    Sp % Not Classified
kNN          0.17  0.74  0.92             None
SVM          0.17  0.75  0.92             None
PLSDA        0.18  0.73  0.92             None
Consensus 1  0.16  0.74  0.93             None
Consensus 2  0.13  0.79  0.96             0.12
